# MCP for a SQL agent — a simple, friendly intro  ·  **simplified**

This notebook builds a tiny **Text2SQL agent**: you ask a question in plain
English, and an LLM uses **database tools** — served over the **Model Context
Protocol (MCP)** — to find the answer in a small SQLite database.

It's the SQL cousin of `v3_mcp_demo_simplified.ipynb`, kept just as small: **one
class** does the whole job, and you start the server yourself.

## ▶ Start the server first

Open a terminal in this folder and run the SQL server, just like the math/text ones:

```bash
python sql_server.py        # serves MCP at http://127.0.0.1:8005/mcp
```

Leave it running. On first launch it creates a small sample database
(`data/sample.db`) and then serves three database tools over HTTP. This notebook is
just the **client** that connects to it.

| Piece | Who plays it | Job |
|-------|--------------|-----|
| **MCP server** | `sql_server.py` (you run it in a terminal) | Owns three database tools and runs them against a SQLite DB. |
| **MCP host** | `MCPHost` (we build it) | Owns the LLM, connects to the server, and lets the model call the tools to answer questions. |

> As in the simpler demo, the app that owns the LLM is the **host**, and the
> per-server **client** role is just a couple of methods folded inside it — one
> class to read, not two.

```
   ┌───────────────────────────────────────────────┐
   │  MCPHost   —   LLM  +  built-in MCP client    │
   └───────────────────────┬───────────────────────┘
                           │
                           │   MCP over HTTP — you started
                           ▼   sql_server.py in a terminal (:8005)
   ┌───────────────────────┴───────────────────────┐
   │  sql_server.py        (HTTP, port 8005)       │
   │     list_tables                               │
   │     get_table_schema    ──►  SQLite           │
   │     execute_sql              data/sample.db   │
   └───────────────────────────────────────────────┘
```

## The sample data

The server seeds three small, related tables (so questions can range from simple
counts to a `JOIN`):

| Table | Rows | What's in it |
|-------|------|--------------|
| `employees` | 8 | name, department, salary, city (3 departments, 3 cities) |
| `customers` | 10 | name, country, currency, balance (4 of them pay in EUR) |
| `orders` | 10 | links to `customers` via `customer_id` (a foreign key) |

## The three tools — and only tools

The server exposes exactly three tools, and **only tools** (no MCP resources or
prompts). The agent uses them in order: **discover → inspect → query**.

| # | Tool | What it does |
|---|------|--------------|
| 1 | `list_tables` | List the tables in the database — call this first. |
| 2 | `get_table_schema` | Show the columns and types of one table. |
| 3 | `execute_sql` | Run a read-only `SELECT` and return the matching rows. |

> Why that order? The LLM can't see the database. It has to *discover* the tables,
> *inspect* the columns, and only then write SQL it knows will actually run —
> instead of guessing names and hallucinating a broken query.

> If a cell below fails to connect, the server isn't running — start it first.

## Step 0 — setup

Imports and configuration. The only slightly unusual lines make sure our calls to
`127.0.0.1` aren't pushed through a corporate proxy, and that Python trusts your
computer's certificates. You can mostly ignore them.

In [8]:
import os
import json
from contextlib import AsyncExitStack

from dotenv import load_dotenv
from openai import OpenAI

# Trust the operating system's certificates (helps behind some corporate networks).
import truststore
truststore.inject_into_ssl()

# The two pieces of the MCP client library we need:
#   ClientSession          -> the MCP conversation (initialize / list_tools / call_tool)
#   streamable_http_client -> the HTTP connection that carries it
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

# Load the OpenAI API key.
load_dotenv("/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Put OPENAI_API_KEY=sk-... in your .env file.")

# Make sure calls to localhost skip any corporate proxy.
for key in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY",
            "http_proxy", "https_proxy", "all_proxy"):
    os.environ.pop(key, None)
os.environ["NO_PROXY"] = "127.0.0.1,localhost,::1"
os.environ["no_proxy"] = "127.0.0.1,localhost,::1"

MODEL = "gpt-5-nano"                            # cheap + fast, good for tool-use demos
SQL_SERVER_URL = "http://127.0.0.1:8005/mcp"    # the server you started in a terminal

print("Setup done. Model:", MODEL)

Setup done. Model: gpt-5-nano


## The server you're running (`sql_server.py`)

We don't build the server in this notebook — you started it in a terminal. But it's
worth knowing what's inside, because it's short. It wraps three plain Python
functions as MCP tools with the `@mcp.tool()` decorator (the decorator turns each
function into a discoverable tool — its name comes from the function, its description
from the docstring, and its JSON schema is generated from the signature), and serves
them over HTTP with `mcp.run(transport="streamable-http")`.

The interesting part is that **safety lives in the tools**, not in the prompt:

* `get_table_schema` validates the table name (rejects anything that isn't a plain
  identifier), and
* `execute_sql` refuses any query that isn't a `SELECT`.

Tools are the boundary between the (non-deterministic) LLM and your real database —
so they check their inputs. Open `sql_server.py` if you'd like to read the whole
thing.

## The whole host: one `MCPHost` class

Here's the entire host. Read it top to bottom — it's short:

* **`connect(name, url)`** — open a connection to the running MCP server, do the MCP
  handshake, and remember every tool it offers (and which session runs it).
* **`call_tool(name, args)`** — run a tool and read back its result.
* **`ask(question)`** — send the question to the LLM with the list of tools. If the
  model asks for a tool, we run it, hand the result back, and loop — until the model
  produces a final answer.
* **`close()`** — close the connection.

The back-and-forth in `ask()` is how an LLM "uses tools": the model never runs any
SQL itself — it just *asks* for a tool by name and supplies the arguments, and
**our** code runs it.

In [9]:
class MCPHost:
    """
    The MCP HOST for our SQL agent: owns the LLM and lets it use the database tools
    from a running MCP server.

    As in the simpler demo, the app that owns the LLM is the host, and the
    per-server "client" role is just a few methods inside it — not a class of its own.
    """

    def __init__(self, model):
        self.model = model
        self.llm = OpenAI()                 # the language model client
        self.stack = AsyncExitStack()       # holds the open connection; closed at the end

        self.tools = []                     # every tool, described in OpenAI's format
        self.tool_session = {}              # tool name -> the MCP session that runs it

    async def connect(self, name, url):
        """Connect to ONE running MCP server and remember the tools it offers."""
        print(f"Connecting to '{name}' server at {url}")

        # Open the HTTP connection, then start an MCP session on top of it.
        read, write, _ = await self.stack.enter_async_context(streamable_http_client(url))
        session = await self.stack.enter_async_context(ClientSession(read, write))
        await session.initialize()          # the MCP handshake

        # Ask the server what tools it has, and register each one.
        tools_response = await session.list_tools()
        for tool in tools_response.tools:
            self.tools.append({
                "type": "function",
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.inputSchema,   # JSON Schema: tells the model the arguments
            })
            self.tool_session[tool.name] = session
            print(f"   found tool: {tool.name}")

    async def call_tool(self, name, arguments):
        """Run a tool and return its text result."""
        session = self.tool_session[name]
        result = await session.call_tool(name, arguments)
        # A tool that returns a list (e.g. list_tables) comes back as several text
        # parts — join them into one string for the model to read.
        return "\n".join(part.text for part in result.content)

    async def ask(self, question):
        """
        Ask a question. The model decides which tools to call and in what order; we
        run them, feed the results back, and loop until it gives a final answer.
        """
        conversation = [{"role": "user", "content": question}]

        while True:
            response = self.llm.responses.create(
                model=self.model,
                instructions=(
                    "You answer questions about a SQLite database using ONLY the tools. "
                    "For each question: (1) call list_tables, (2) call get_table_schema "
                    "on the relevant table(s), (3) write one SQLite SELECT query, "
                    "(4) run it with execute_sql. Never guess table or column names — "
                    "confirm them with the tools first. Then give a short, plain answer."
                ),
                input=conversation,
                tools=self.tools,
            )

            # Did the model ask to call any tools this turn?
            tool_calls = [item for item in response.output if item.type == "function_call"]
            if not tool_calls:
                return response.output_text         # no tools -> this is the final answer

            for call in tool_calls:
                arguments = json.loads(call.arguments)   # model sends args as a JSON string
                print(f"   model wants: {call.name}({arguments})")

                result = await self.call_tool(call.name, arguments)

                # Keep the trace readable: show the result on one line, shortened if long.
                oneline = " ".join(result.split())
                print(f"      tool says: {oneline[:90]}{'...' if len(oneline) > 90 else ''}")

                # Record the call and its result so the model can use them next turn.
                conversation.append({
                    "type": "function_call",
                    "call_id": call.call_id,
                    "name": call.name,
                    "arguments": call.arguments,
                })
                conversation.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": result,
                })

    async def close(self):
        """Close the server connection."""
        await self.stack.aclose()

### Try it by hand — connect and call the tools (no AI yet)

First, no LLM at all. We connect to the running server, see the tools we
discovered, and call each one ourselves — the same **discover → inspect → query**
the agent will do. We also try a forbidden write to show the safety guard kick in.

> Reminder: an MCP connection must be **opened and closed in the same notebook
> cell**, so each demo connects at the top and closes at the bottom.

In [10]:
host = MCPHost(model=MODEL)
await host.connect("sql", SQL_SERVER_URL)

print("\nTools the host can use now:")
for tool in host.tools:
    print(f"  - {tool['name']}: {tool['description']}")

print("\n1) list_tables():")
print(await host.call_tool("list_tables", {}))

print("\n2) get_table_schema('customers'):")
print(await host.call_tool("get_table_schema", {"table_name": "customers"}))

print("\n3) execute_sql(... count EUR customers ...):")
print(await host.call_tool(
    "execute_sql",
    {"query": "SELECT COUNT(*) AS eur_customers FROM customers WHERE currency = 'EUR'"},
))

print("\n4) a blocked write (execute_sql refuses anything but SELECT):")
print(await host.call_tool("execute_sql", {"query": "DELETE FROM customers"}))

# Close in this same cell.
await host.close()
print("\nDone — connection closed.")

Connecting to 'sql' server at http://127.0.0.1:8005/mcp
   found tool: list_tables
   found tool: get_table_schema
   found tool: execute_sql

Tools the host can use now:
  - list_tables: Get all table names in the database. Call this FIRST to discover what tables exist before writing any query.
  - get_table_schema: Get the column names and types for one table. Use this after list_tables, before writing SQL, so you only reference columns that really exist.
  - execute_sql: Run a read-only SQL SELECT query and return the matching rows as text. ONLY SELECT is allowed — no INSERT, UPDATE, DELETE, DROP, or ALTER.

1) list_tables():
customers
employees
orders

2) get_table_schema('customers'):
{
  "name": "id",
  "type": "INTEGER",
  "nullable": true,
  "primary_key": true
}
{
  "name": "name",
  "type": "TEXT",
  "nullable": false,
  "primary_key": false
}
{
  "name": "country",
  "type": "TEXT",
  "nullable": false,
  "primary_key": false
}
{
  "name": "currency",
  "type": "TEXT",
  "nu

### Now let the LLM drive

Same host, but now we just ask questions in English and let the model decide which
tools to call. Watch it chain them — `list_tables` → `get_table_schema` →
`execute_sql` — and, for the last question, inspect **two** tables and write a
`JOIN`.

In [11]:
host = MCPHost(model=MODEL)
await host.connect("sql", SQL_SERVER_URL)

try:
    questions = [
        "How many customers pay in EUR?",
        "What is the average salary in each department?",
        "Which products were ordered by customers in Germany?",
    ]
    for question in questions:
        print("\n" + "=" * 72)
        print("Q:", question)
        answer = await host.ask(question)
        print("\nA:", answer)
finally:
    # Always close, even if a question errors — still in this same cell.
    await host.close()

Connecting to 'sql' server at http://127.0.0.1:8005/mcp
   found tool: list_tables
   found tool: get_table_schema
   found tool: execute_sql

Q: How many customers pay in EUR?
   model wants: list_tables({})
      tool says: customers employees orders
   model wants: get_table_schema({'table_name': 'customers'})
      tool says: { "name": "id", "type": "INTEGER", "nullable": true, "primary_key": true } { "name": "name...
   model wants: execute_sql({'query': "SELECT COUNT(*) AS eur_customer_count FROM customers WHERE currency = 'EUR';"})
      tool says: eur_customer_count ------------------ 4

A: 4

Q: What is the average salary in each department?
   model wants: list_tables({})
      tool says: customers employees orders
   model wants: get_table_schema({'table_name': 'employees'})
      tool says: { "name": "id", "type": "INTEGER", "nullable": true, "primary_key": true } { "name": "name...
   model wants: execute_sql({'query': 'SELECT department, AVG(salary) AS average_salary FROM

---
## Recap

* One **`MCPHost`** connected to the server you started, handed its tools to an LLM,
  and ran the tool-calling loop — the whole agent in a single class.
* The server ran **independently over HTTP** (you launched `sql_server.py` in a
  terminal); the notebook was just a client that connected to it.
* The agent answered each question by **chaining tools**: `list_tables` →
  `get_table_schema` → `execute_sql`. The LLM chose the order; our code just ran
  what it asked for.
* Safety lived in the **tools**, not the prompt: `execute_sql` refuses anything but
  `SELECT`, and `get_table_schema` validates the table name.

### Try next
* Add a fourth tool to `sql_server.py` (say, `row_count(table_name)`), restart the
  server, and re-run **Demo 1** — it appears automatically, no change to `MCPHost`.
* Ask something that needs a `JOIN` across two tables and watch the tool chain grow.
* Notice this notebook is almost identical to `v3_mcp_demo_simplified.ipynb` — same
  host, different tools. That's the point of MCP: the host doesn't care what the
  tools do.

# Now Let's use Langchain SDK

In [12]:
%pip install langchain-mcp-adapters

  Using cached langchain_mcp_adapters-0.3.2-py3-none-any.whl.metadata (11 kB)
Using cached langchain_mcp_adapters-0.3.2-py3-none-any.whl (28 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from IPython.display import Image, display

# The question this notebook asks of every agent it builds.
BUSINESS_QUESTION = "What was our total revenue, excluding cancelled orders?"

# One instruction string, shared by all three frameworks, so the comparison in P9 is fair.
# The tool ORDER is spelled out deliberately: left to itself the model will guess a plausible
# table name, query it, and report that the data is missing. Naming the discovery steps costs
# one sentence and removes that whole failure mode.
AGENT_INSTRUCTIONS = (
    "You are InsightAgent, a data analyst for an online-retail store. "
    "Always work in this order: first list the tables, then inspect the schema of every table "
    "you intend to use, and only then write SQL. Never guess a table or column name. "
    "Revenue must EXCLUDE cancelled invoices (invoices.is_cancelled = 1). "
    "State the final answer clearly, including the number."
)

# LangChain's MCP client. Same server the OpenAI SDK cell below uses —
# `transport` + `url` is all it needs (no command, no args).
client = MultiServerMCPClient(
    {
        "sql": {
            "transport": "streamable_http",
            "url": SQL_SERVER_URL,
        }
    }
)

# Ask the server for its tools; each MCP tool comes back as a LangChain tool.
tools = await client.get_tools()
print("Tools from the MCP server:", [tool.name for tool in tools])

prebuilt_agent = create_agent(
    model="openai:gpt-4.1-mini",
    system_prompt=AGENT_INSTRUCTIONS,
    tools=tools,
)


Tools from the MCP server: ['list_tables', 'get_table_schema', 'execute_sql']


In [14]:
questions = [
    "How many customers pay in EUR?",
    "What is the average salary in each department?",
    "Which products were ordered by customers in Germany?",
]

for i, q in enumerate(questions, 1):
    print(f"\n────────── Test {i} ──────────")
    print(f"❓ {q}")
    result = await prebuilt_agent.ainvoke({"messages": [{"role": "user", "content": q}]})
    print(f"💬 {result['messages'][-1].content}")



────────── Test 1 ──────────
❓ How many customers pay in EUR?
💬 There are 4 customers who pay in EUR.

────────── Test 2 ──────────
❓ What is the average salary in each department?
💬 The average salary in each department is as follows:
- Engineering: 95,500.0
- Marketing: 70,000.0
- Sales: 60,000.0

────────── Test 3 ──────────
❓ Which products were ordered by customers in Germany?
💬 The products ordered by customers in Germany are: Widget A, Widget B, and Gadget Y.


# Now Let's use Open AI SDK

In [15]:
from agents import Agent, Runner                                                             
from agents.mcp import MCPServerStreamableHttp                                               
                                                                                            
SQL_URL = "http://127.0.0.1:8005/mcp"                                                       



async def agents_sdk_http_demo():
    # Note: `params={"url": ...}` — no command, no args. The SDK just opens                  
    # a streamable-http session to the URL. Servers are fully independent.                   
    sql_mcp = MCPServerStreamableHttp(                                                      
        params={"url": SQL_URL},                                                            
        cache_tools_list=True,                                                               
    )                                                                                        
    
                                                                                            
    # `async with` still manages the CLIENT session lifecycle,                               
    # but the server processes keep running after this block exits.
    async with sql_mcp:                                                           
        agent = Agent(                                                                     
            name="MCP Demo Assistant",                                                       
            model=MODEL,                                                                   
            instructions=(                                                                   
                "You have access to a SQL database over MCP. "
                "Use it for every operation — do not compute anything yourself, "          
                "even trivial values. Chain tools when needed."                            
            ),                                                                               
            mcp_servers=[sql_mcp],
        )                                                                                    
                                                                                            
        questions = [
            "How many customers pay in EUR?",
            "What is the average salary in each department?",
            "Which products were ordered by customers in Germany?",                   
        ]                                                                                    

        for i, q in enumerate(questions, 1):                                                 
            print(f"\n────────── Test {i} ──────────")                                     
            print(f"❓ {q}")
            result = await Runner.run(starting_agent=agent, input=q)                         
            print(f"💬 {result.final_output}")
                                                                                            
                                                                                            
await agents_sdk_http_demo()                                    


────────── Test 1 ──────────
❓ How many customers pay in EUR?
💬 4 customers pay in EUR. If you want, I can list their names or IDs.

────────── Test 2 ──────────
❓ What is the average salary in each department?
💬 Here are the average salaries by department (based on the employees table):

- Engineering: 95,500.00
- Marketing: 70,000.00
- Sales: 60,000.00

If you want the exact SQL or additional stats (e.g., count per department), I can provide that.

────────── Test 3 ──────────
❓ Which products were ordered by customers in Germany?
💬 Here are the products ordered by customers in Germany (distinct products):

- Widget A
- Widget B
- Gadget Y

Would you like counts or total quantities per product for Germany as well?
